# Recorte de enunciados de PDFs de emestrada — paso a paso

In [1]:
# %pip install pdfplumber img2pdf pillow pandas

import io
import re
import os
import shutil
from pathlib import Path
import unicodedata

import pandas as pd
import pdfplumber
import img2pdf
from PIL import Image
from IPython.display import display

Establecer directorio raiz

In [2]:
try:
    root = "/home/daniel/code/clasificador_selectividad"
    os.chdir(root)
except Exception as e:
    root = "/Users/daniel/code/clasificador_selectividad"
    os.chdir(root)

Parámetros para medidas de recuadros y pdf

In [3]:
ANCHO_MIN, ALTO_MIN = 300, 20   # tamaño mínimo para considerar un rectángulo como recuadro de enunciado
MARGEN = 2                      # holgura alrededor del borde al recortar
DPI = 200                       # resolución de los PNG
A4 = (595.28, 841.89)
MARGEN_A4 = 40                  # margen de la hoja de salida
SEPARACION = 15                 # hueco entre recuadros apilados

Función para detectar los recuadros

In [4]:
def detect_rectangles(page):
    vistos = {}
    for r in page.rects:
        if r["width"] > ANCHO_MIN and r["height"] > ALTO_MIN:
            clave = tuple(round(r[k]) for k in ("x0", "top", "x1", "bottom"))
            vistos[clave] = r
    return sorted(vistos.values(), key=lambda r: r["top"])

In [5]:
# PATRON = re.compile(
#     r"""
#     (?P<asig>SOCIALES[ \t]+(?:II|2)|MATEM[AÁ]TICAS[ \t]+(?:II|2)|QU[IÍ]MICA|F[IÍ]SICA)
#     [ \t.,:;-]*
#     (?P<anio>(?:19|20)\d{2})
#     [ \t.,:;-]*
#     (?P<conv>(?:JUNIO|JULIO|SEPTIEMBRE|PONENCIA|RESERVA|MODELO)(?:[ \t]*\d+)?)
#     [ \t.,:;-]*
#     (?P<ej>[^\n\r]*?)
#     [ \t.]*\r?$
#     """,
#     re.IGNORECASE | re.MULTILINE | re.VERBOSE,
# )

In [6]:
def norm(s: str) -> str:
    s = unicodedata.normalize("NFKD", s.upper())
    s = "".join(c for c in s if not unicodedata.combining(c))  # quita tildes
    return re.sub(r"\s+", "", s)

Extraer y procesar los datos del examen

In [7]:
def process_exercise_info(exercise_info):
    exercise_info = exercise_info.lower()

    # valores por defecto si no puede procesarse
    # la información del ejercicio
    (asignatura, year, convocatoria, ejercicio) = ("" for _ in range(4))
    
    asignaturas = {
        "quimica" : "Química",
        "química" : "Química",
        "física" : "Física",
        "fisica" : "Física",
        "sociales ii" : "Mates CCSS",
        "matematicas ii" : "Matemáticas II"
    }

    # extraer asignatura
    for k,v in asignaturas.items():
        if k in exercise_info:
            asignatura = v
            break
        else:
            asignatura = "asignatura_desconocida"

    # extraer año
    year_pattern = r"2\d{3}"
    year = re.findall(year_pattern, exercise_info)[0]

    # extraer convocatoria y ejercicio
    convocatorias = ["junio", "julio", "septiembre",
                     "modelo", "ponencia",
                     "reserva"]

    # convocatorias de Junio, Julio, Septiembre, modelo o ponencia
    for conv in convocatorias[:-2]:
        if conv in exercise_info:
            convocatoria = conv.title()
            convocatoria_position = exercise_info.find(conv)
            ejercicio = (" ".join(exercise_info[convocatoria_position:].split(" ")[1:])
                         .title()
            )
            break

    # convocatoria es una reserva
    if "reserva" in exercise_info:
        convocatoria_position = exercise_info.find("reserva")
        convocatoria = (" ".join(exercise_info[convocatoria_position:]
                                 .split(" ")[:2])
                                 .title()
                                 .replace(".", "")
        )
        ejercicio = (" ".join(
            exercise_info[convocatoria_position:].split(" ")[2:]
            ).title()
        )
    
    return (asignatura, year, convocatoria, ejercicio)   


Limpieza de texto

In [8]:
def slug(texto):
    texto = unicodedata.normalize("NFKD", texto).encode("ascii", "ignore").decode()
    return re.sub(r"\W+", "_", texto).strip("_").lower()

#print(slug("RESERVA 1"), "|", slug("SEPTIEMBRE"), "|", slug("2023 - Integrales"))

Composición del PDF de enunciados a partir de las imágenes (`img2pdf`).

Se rasteriza para que el PDF no arrastre el contenido oculto de la página original (soluciones).

In [ ]:
def _a_png(img, dpi):
    buf = io.BytesIO()
    img.save(buf, format="PNG", dpi=(dpi, dpi))  # el dpi embebido fija el tamaño físico en img2pdf
    return buf.getvalue()


def componer_pdf(imagenes, salida, dpi):
    """Apila las imágenes en hojas A4, una debajo de otra, saltando de hoja cuando no caben."""
    px = dpi / 72                                   # puntos -> píxeles
    ancho, alto = round(A4[0] * px), round(A4[1] * px)
    margen, sep = round(MARGEN_A4 * px), round(SEPARACION * px)
    util_w, util_h = ancho - 2 * margen, alto - 2 * margen

    paginas, hoja, y = [], None, None
    for img in imagenes:
        escala = min(1.0, util_w / img.width, util_h / img.height)
        if escala < 1:
            img = img.resize((round(img.width * escala), round(img.height * escala)), Image.LANCZOS)
        if hoja is None or y + img.height > alto - margen:
            if hoja is not None:
                paginas.append(_a_png(hoja, dpi))
            hoja = Image.new("RGB", (ancho, alto), "white")
            y = margen
        hoja.paste(img, ((ancho - img.width) // 2, y))
        y += img.height + sep
    paginas.append(_a_png(hoja, dpi))

    salida.write_bytes(img2pdf.convert(paginas))

Procesamiento de un archivo pdf con enunciados

In [9]:
def procesar_pdf(ruta_pdf, output_folder, export_pdf = True, export_png=True, dpi=DPI):
    exercises, imagenes = [], []
    # ruta_pdf
    # 'clasificador_selectividad/source_pdf_files/Física - Temas/campo_eléctrico_magnético/2016 - Campo eléctrico y magnético.pdf'
    path_parts = ruta_pdf.parts
    subject_path = path_parts[-3].replace(" - Temas", "")
    topic_path = path_parts[-2]
    png_folder = output_folder / "png_enunciados" / f"png_{subject_path}_{ruta_pdf.stem}"

    # 1) Detección con pdfplumber
    with pdfplumber.open(ruta_pdf) as pdf:
        # procesamiento de las páginas del pdf
        for n, page in enumerate(pdf.pages):
            # detectar los recuadros
            boxes = detect_rectangles(page)

            # obtener coordenadas del recuadro
            for k, r in enumerate(boxes, start=1):
                x0 = max(r["x0"] - MARGEN, 0)
                x1 = min(r["x1"] + MARGEN, page.width)
                top = max(r["top"] - MARGEN, 0)
                bottom = min(r["bottom"] + MARGEN, page.height)

                # recortar el trozo del recuadro del pdf
                clip = page.crop((x0, top, x1, bottom))

                # extraer el texto del enunciado en
                # el recuadro
                texto_raw = clip.extract_text() or ""

                # separar el enunciado en líneas
                lineas = texto_raw.split("\n")

                # el enunciado son todas las líneas excepto
                # la última (información del ejercicio)
                texto = "\n".join(lineas[:-1])

                # información del ejercicio
                exercise_info = lineas[-1]
                asignatura, anio, conv, ej = process_exercise_info(exercise_info)
                # texto = texto_raw.replace("\n", " ")
                # metadata = PATRON.search(texto_raw)   # buscar con saltos de línea

                # crear nombre de archivo png
                # a partir de la información del ejercicio
                if exercise_info:
                    ej_slug = slug(ej)
                    png_name = f"{asignatura}_{topic_path}_{anio}_{slug(conv)}_{ej_slug}"
                    # ej_slug = slug(re.sub(r"EJERCICIO", "", metadata["ej"], flags=re.IGNORECASE))
                    # nombre = f"{metadata['anio']}_{slug(metadata['conv'])}_{ej_slug}"
                else:
                    png_name = f"{slug(ruta_pdf.stem)}_p{n + 1:02d}"

                # rasterizar el recorte (se usa para el png y para el pdf)
                imagen_pil = None
                if export_png or export_pdf:
                    imagen_pil = clip.to_image(resolution=dpi).original.convert("RGB")

                # exportar recorte a png
                image = None
                if export_png:
                    png_folder.mkdir(parents=True, exist_ok=True)
                    output_file = png_folder / f"{png_name}.png"
                    i = 2
                    while output_file.exists():
                        output_file = png_folder / f"{png_name}_{i}.png"
                        i += 1
                    imagen_pil.save(output_file, dpi=(dpi, dpi))
                    image = str(output_file.relative_to(output_folder))

                if export_pdf:
                    imagenes.append(imagen_pil)

                aviso = ""
                if not exercise_info:
                    aviso = "sin info ejercicio"
                elif len(boxes) > 1:
                    aviso = f"{len(boxes)} recuadros en la página"

                # asignatura y tema
                #asignatura = ruta_pdf.parts[-3].replace(" - Temas", "").replace(" ", "").lower()
                tema = ruta_pdf.stem.split(" - ")[1]
                if asignatura == "asignatura_desconocida":
                    asignatura = subject_path
                    
                exercises.append({
                    "asignatura": asignatura,
                    "asignatura_carpeta" : subject_path,
                    "año" : anio,
                    "convocatoria" : conv,
                    "ejercicio" : ej,
                    "tema": tema,
                    "info_ejercicio" : exercise_info,
                    "enunciado": texto,
                    "archivo": ruta_pdf.name,
                    "imagen": image,
                    "aviso": aviso,
                    #"año": metadata["anio"] if metadata else "",
                    #"convocatoria": metadata["conv"].title() if metadata else "",
                    #"ejercicio": metadata["ej"] if metadata else "",
                })

    if not exercises:
        return pd.DataFrame(), "ningún recuadro detectado"

    # combinar todos los enunciados en un pdf (raster, desde las imágenes)
    if export_pdf:
        carpeta_pdf = output_folder / "pdf_enunciados"
        carpeta_pdf.mkdir(parents=True, exist_ok=True)
        output_pdf_filename = carpeta_pdf / f"{subject_path}_{ruta_pdf.stem}_enunciados.pdf"
        componer_pdf(imagenes, output_pdf_filename, dpi)

    return pd.DataFrame(exercises), ""

## Procesar carpeta

In [10]:
def process_folder(folder, export_pdf = True, export_png = True, files_to_process_limit=None):

    # '/home/daniel/code/clasificador_selectividad/source_pdf_files/Física - Temas/campo_gravitatorio'
    source_folder = Path(folder)
    output_folder_name = "/".join(source_folder.parts[-2:])
    output_folder = Path(f"{root}/solo_enunciados_ejercicios/{output_folder_name}")
    output_folder.mkdir(parents=True, exist_ok=True)
    print(f"Procesando carpeta: {output_folder_name}")

    # obtener rutas de los archivos pdf de la carpeta
    pdf_files = sorted(p for p in source_folder.rglob("*.pdf") if not p.stem.endswith("_enunciados"))
    
    # hay límite de archivos por procesar
    if files_to_process_limit:
        pdf_files = pdf_files[:files_to_process_limit]

    print(f"{len(pdf_files)} PDFs a procesar\n")

    dataframes, issues = [], []
    for index, file_path in enumerate(pdf_files, start=1):
        try:
            df, aviso = procesar_pdf(file_path, output_folder, export_pdf = export_pdf, export_png = export_png)
        except Exception as e:
            subject = file_path.parts[-3].replace(" - Temas", "")
            issues.append({"issue": f"ERROR: {e}",
                           "subject" : subject,
                           "pdf file": str(file_path.name)})
            print(f"[{index}/{len(pdf_files)}] ERROR  ({subject}) {file_path.name}: {e}")
            continue

        dataframes.append(df)

        if aviso:
            issues.append({"issue": aviso,
                           "pdf file": str(file_path)})
            
            print(f"[{index}/{len(pdf_files)}] AVISO  {file_path.name}: {aviso}")
        else:
            print(f"[{index}/{len(pdf_files)}] OK     {file_path.name}: {len(df)} ejercicios procesados")

    # Guardar resultados en CSV
    enunciados_df = pd.DataFrame()
    for d in dataframes:
        enunciados_df = pd.concat([enunciados_df, d])
            
    enunciados_df.to_csv(output_folder / f"{output_folder.name}.csv", index=False)

    if issues:
        issues_df = pd.DataFrame(issues)
        issues_df.to_csv(output_folder / f"{output_folder.name}_issues.csv",
                            index=False, encoding="utf-8")

    print("*" * 40)
    print(f"\nProcesamiento completado. Resultados guardados en: {output_folder}\n\n")

## Procesar carpetas

Normalizar texto de nombres de temas y carpetas

In [11]:
def normalize_text(texto):
    descompuesto = unicodedata.normalize("NFD", texto)
    return "".join(c for c in descompuesto if not unicodedata.combining(c))

Obtener las rutas de los archivos

In [12]:
os.chdir(f"{root}/source_pdf_files")

In [13]:
carpetas_asignaturas = {d.name:[] for d in Path(".").iterdir() if d.is_dir()}

In [14]:
for asignatura in carpetas_asignaturas:
    carpetas_asignaturas[asignatura] = [d.name for d in Path(f"./{asignatura}").iterdir() if d.is_dir()]

In [15]:
folder_paths = []
for asignatura, carpetas in carpetas_asignaturas.items():
    for carpeta in carpetas:
        folder_paths.append(f"{root}/source_pdf_files/{asignatura}/{carpeta}")

In [16]:
# folder_paths

# PROCESAR LOS ARCHIVOS

In [17]:
# process_folder(folder_paths[0], files_to_process_limit=None)

In [18]:
for folder in folder_paths:
    folder_path = Path(folder)
    # Limitar a 5 archivos por carpeta para pruebas
    process_folder(folder, export_pdf = True, export_png = False, files_to_process_limit=None)

Procesando carpeta: Física - Temas/campo_eléctrico_magnético
10 PDFs a procesar

[1/10] OK     2016 - Campo eléctrico y magnético.pdf: 10 ejercicios procesados
[2/10] OK     2017 - Campo eléctrico y magnético.pdf: 12 ejercicios procesados
[3/10] OK     2018 - Campo eléctrico y magnético.pdf: 12 ejercicios procesados
[4/10] OK     2019 - Campo eléctrico y magnético.pdf: 12 ejercicios procesados
[5/10] OK     2020 - Campo eléctrico y magnético.pdf: 12 ejercicios procesados
[6/10] OK     2021 - Campo eléctrico y magnético.pdf: 12 ejercicios procesados
[7/10] OK     2022 - Campo eléctrico y magnético.pdf: 12 ejercicios procesados
[8/10] OK     2023 - Campo eléctrico y magnético.pdf: 12 ejercicios procesados
[9/10] OK     2024 - Campo eléctrico y magnético.pdf: 12 ejercicios procesados
[10/10] OK     2025 - Campo eléctrico y magnético.pdf: 15 ejercicios procesados
****************************************

Procesamiento completado. Resultados guardados en: /Users/danie

## Mover archivos .csv

In [19]:
os.chdir(root)
Path("csv_enunciados").mkdir(exist_ok=True)
os.chdir("solo_enunciados_ejercicios")

In [20]:
for folder in Path(".").iterdir():
    if folder.is_dir():
        subject = folder.name.split(" - ")[0].replace(" ", "_").lower()
        csv_files = list(folder.rglob("*.csv"))
        for csv_file in csv_files:
            new_filename = f"{subject}_{csv_file.stem}.csv"
            if "_issues" in csv_file.name:
                Path(f"{root}/csv_enunciados/issues").mkdir(exist_ok=True)
                output_csv_path = Path(f"{root}/csv_enunciados/issues/{subject}_{csv_file.stem}_issues.csv")
            else:
                new_filename = f"{subject}_{csv_file.stem}_enunciados.csv"
                output_csv_path = Path(f"{root}/csv_enunciados/{new_filename}")

            shutil.move(csv_file, output_csv_path)

Combinar todos los issues

In [21]:
issue_files_path = Path(root) / "csv_enunciados" / "issues"
issue_files = list(issue_files_path.iterdir())
if issue_files:
    issue_df = pd.DataFrame()
    for f in issue_files:
        if f.suffix == ".csv":
            df = pd.read_csv(f, encoding="utf-8")
            issue_df = pd.concat([issue_df, df])
    issue_df.to_csv(issue_files_path /  "../issues.csv", index = False )

# clean issue files
for f in issue_files:
    f.unlink()

# borrar directorio issues
shutil.rmtree(f"{root}/csv_enunciados/issues")

In [22]:
issue_df

,issue,subject,pdf file
0,ERROR: list index out of range,Mates CCSS,2016 - Funciones CCSS.pdf
0,ERROR: list index out of range,Química,2021 - Enlace Químico.pdf
0,ERROR: list index out of range,Mates II,2019 - Integrales.pdf
0,ningún recuadro detectado,NaN,/Users/daniel/code/clasificador_selectividad/s...
0,ERROR: list index out of range,Química,2008 - Reactividad Orgánica.pdf
0,ERROR: list index out of range,Mates CCSS,2007 - Probabilidad.pdf
1,ERROR: list index out of range,Mates CCSS,2009 - Probabilidad.pdf
0,ERROR: list index out of range,Química,2001 - Equilibrio Químico.pdf
1,ERROR: list index out of range,Química,2002 - Equilibrio Químico.pdf


Combinar en un único CSV

In [23]:
os.chdir(f"{root}/csv_enunciados")

In [24]:
csv_files = [i for i in Path(".").rglob("*.csv") if i.name != "issues.csv"]

In [25]:
enunciados_df = pd.DataFrame()
failed_files = []
for f in csv_files:
    try:
        df = pd.read_csv(f, encoding="utf-8")
        enunciados_df = pd.concat([enunciados_df, df])
    except Exception as e:
        failed_files.append(f)
        print(f"Error: {f.name} | {e}")
enunciados_df.to_csv("todos_enunciados.csv", encoding="utf-8", index = False)

for f in csv_files:
    if f not in failed_files:
        f.unlink()

In [30]:
todos_df = pd.read_csv("todos_enunciados.csv")
todos_df.sample(10)

,asignatura,asignatura_carpeta,año,convocatoria,ejercicio,tema,info_ejercicio,enunciado,archivo,imagen,aviso
2009,Mates CCSS,Mates CCSS,2001,Reserva 1,Ejercicio 2. Opción A,Funciones CCSS,SOCIALES II. 2001 RESERVA 1. EJERCICIO 2. OPCI...,Un agricultor comprueba que si el precio al qu...,2001 - Funciones CCSS.pdf,NaN,NaN
4364,Mates II,Mates II,2024,Julio,Ejercicio 2,Funciones,MATEMÁTICAS II. 2024. JULIO. EJERCICIO 2,Sea la función f : → dada por f ( x ) =\n\n...,2024 - Funciones.pdf,NaN,NaN
4072,Mates II,Mates II,2000,Junio,Ejercicio 2. Opción B.,Funciones,MATEMÁTICAS II. 2000. JUNIO. EJERCICIO 2. OPCI...,"a\nDetermina a, b, c, para que la curva f(x) ...",2000 - Funciones.pdf,NaN,NaN
1851,Mates II,Mates II,2022,Julio,Ejercicio 8,Geometría,MATEMÁTICAS II. 2022. JULIO. EJERCICIO 8,y\nSea el plano  x yz  2 y la recta r  x...,2022 - Geometría.pdf,NaN,NaN
528,Química,Química,2007,Reserva 3,Ejercicio 6. Opción A,Ácido Base,QUÍMICA. 2007. RESERVA 3. EJERCICIO 6. OPCIÓN A,"Se disuelven 23 g de ácido metanoico, HCOOH, e...",2007 - Ácido Base.pdf,NaN,NaN
444,Mates CCSS,Mates CCSS,2024,Modelo,Ejercicio 3A,Probabilidad,SOCIALES II. 2024 MODELO. EJERCICIO 3A,"Para tratar cierta enfermedad, en un hospital ...",2024 - Probabilidad.pdf,NaN,NaN
736,Mates CCSS,Mates CCSS,2003,Septiembre,Ejercicio 3 Parte Ii Opción A,Inferencia Estadística,SOCIALES II. 2003 SEPTIEMBRE. EJERCICIO 3 PART...,Se sabe que la antigüedad de los coches fabric...,2003 - Inferencia Estadística.pdf,NaN,NaN
606,Química,Química,2015,Reserva 4,Ejercicio 4. Opción B,Ácido Base,QUÍMICA. 2015. RESERVA 4. EJERCICIO 4. OPCIÓN B,Se tienen dos disoluciones acuosas de la misma...,2015 - Ácido Base.pdf,NaN,NaN
2805,Mates CCSS,Mates CCSS,2023,Reserva 1,Ejercicio A1,Matrices Y Determinantes,SOCIALES II. 2023 RESERVA 1. EJERCICIO A1,a) Un agricultor vende la producción de tres t...,2023 - Matrices Y Determinantes.pdf,NaN,NaN
583,Química,Química,2013,Reserva 1,Ejercicio 6. Opción A,Ácido Base,QUÍMICA. 2013. RESERVA 1. EJERCICIO 6. OPCIÓN A,"Tenemos una disolución 0,05 M de ácido benzoic...",2013 - Ácido Base.pdf,NaN,NaN


In [27]:
todos_df.asignatura.value_counts()

asignatura
Química       1785
Mates II      1255
Mates CCSS    1189
Física         486
Name: count, dtype: int64

In [28]:
todos_df[todos_df.asignatura == "asignatura_desconocida"]

,asignatura,asignatura_carpeta,año,convocatoria,ejercicio,tema,info_ejercicio,enunciado,archivo,imagen,aviso


In [32]:
todos_df[todos_df.enunciado == ""]

,asignatura,asignatura_carpeta,año,convocatoria,ejercicio,tema,info_ejercicio,enunciado,archivo,imagen,aviso


In [34]:
todos_df.convocatoria.unique()

<StringArray>
[             'Junio',          'Reserva 1',          'Reserva 2',
          'Reserva 3',          'Reserva 4',         'Septiembre',
              'Julio',             'Modelo',                  nan,
 'Reserva1 Ejercicio']
Length: 10, dtype: str

In [35]:
todos_df[todos_df.convocatoria == "Reserva1 Ejercicio"]

,asignatura,asignatura_carpeta,año,convocatoria,ejercicio,tema,info_ejercicio,enunciado,archivo,imagen,aviso
1263,Física,Física,2020,Reserva1 Ejercicio,1,Campo gravitatorio,FISICA. 2020. RESERVA1. EJERCICIO 1,a) Considere dos partículas de igual masa sepa...,2020 - Campo gravitatorio.pdf,NaN,NaN


In [37]:
todos_df.loc[0, :]

asignatura                                                      Química
asignatura_carpeta                                             Química
año                                                                2000
convocatoria                                                      Junio
ejercicio                                         Ejercicio 2. Opción B
tema                                                 Conf. Electrónica
info_ejercicio              QUÍMICA. 2000. JUNIO. EJERCICIO 2. OPCIÓN B
enunciado             Los números atómicos de los elementos P y Mn s...
archivo                                   2000 - Conf. Electrónica.pdf
imagen                                                              NaN
aviso                                                               NaN
Name: 0, dtype: object

In [38]:
todos_df.columns

Index(['asignatura', 'asignatura_carpeta', 'año', 'convocatoria', 'ejercicio',
       'tema', 'info_ejercicio', 'enunciado', 'archivo', 'imagen', 'aviso'],
      dtype='str')